# Fine-tune transfer — epochs to reach a target accuracy

Same *epochs-to-target* analysis as the from-scratch notebook, but for the
`finetune_transfer_20260601` sweep. Here each held-out task is fine-tuned (its
new `p_vector` row, plus extra param groups per condition) on top of a frozen
leave-one-out model. The per-epoch test accuracy of the held-out task lives in
`history.npz` (key `test_accs`, 200 epochs).

One table per **(batch size × condition)** = 3 × 5 = 15 tables. For the
`p only` condition we use the `new` (random) init-source; the other four
conditions use `random_peer`. Within each table, each row is a held-out task
and we report the first epoch its test accuracy crosses `TARGET_ACC`, across
seeds 0–3 (`NaN` / `n_seeds_reached` < 4 where it never reaches the target).

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display, Markdown

ROOT = Path(r"C:\Users\garcias\Downloads\finetune_transfer_20260601")
TARGET_ACC = 0.8  # <-- change to the % you care about (e.g. 0.5, 0.9)
SEEDS = [0, 1, 2, 3]

BATCH_DIRS = {
    16: "20260601_16_finetune_transfer",
    32: "20260601_32_finetune_transfer",
    64: "20260601_64_finetune_transfer",
}

# condition_slug -> (display name, init-source folder to use)
CONDITIONS = {
    "p_vector": ("p only", "new"),
    "p_p2C": ("p + p2C", "random_peer"),
    "p_p2D": ("p + p2D", "random_peer"),
    "p_p2C_p2D": ("p + p2C + p2D", "random_peer"),
    "p_p2A_p2W": ("p + p2A + p2W", "random_peer"),
}

# Held-out tasks, in TASKS_MAP order.
TASKS = [
    "NoiseCleaner", "DelayPro", "DelayAnti", "CatPro", "CatAnti",
    "Match2Sample", "NonMatch2Sample", "CtxIntMod1", "CtxIntMod2",
    "ArithMultiply", "ArithAdd", "CopyTask", "GoNogo", "PerceptualDM",
    "DelayedComparison", "DurationPro", "DurationAnti", "IntDisc", "MultiSens",
]

assert ROOT.is_dir(), f"Not found: {ROOT}"
print("Root:", ROOT)

Root: C:\Users\garcias\Downloads\finetune_transfer_20260601


In [2]:
def load_curve(bs, cond_slug, init, task, seed):
    """Per-epoch test-accuracy curve for one run, or None if missing."""
    p = (ROOT / BATCH_DIRS[bs] / f"seed_{seed}" / task / cond_slug / init
         / "history.npz")
    if not p.is_file():
        return None
    return np.asarray(np.load(p, allow_pickle=True)["test_accs"], dtype=float)


def first_crossing(curve, target):
    """First epoch index where curve >= target, or None if never reached."""
    if curve is None:
        return None
    hit = np.where(curve >= target)[0]
    return int(hit[0]) if hit.size else None


def epochs_to_target(bs, cond_slug, init, target):
    rows = []
    for task in TASKS:
        per_seed_epoch = {
            seed: first_crossing(load_curve(bs, cond_slug, init, task, seed), target)
            for seed in SEEDS
        }
        reached = [e for e in per_seed_epoch.values() if e is not None]
        rows.append(
            {
                "task": task,
                "n_seeds_reached": len(reached),
                "mean_epoch": np.mean(reached) if reached else np.nan,
                "std_epoch": np.std(reached) if reached else np.nan,
                **{f"seed_{s}": per_seed_epoch[s] for s in SEEDS},
            }
        )
    return (
        pd.DataFrame(rows)
        .set_index("task")
        .sort_values(["n_seeds_reached", "mean_epoch"], ascending=[False, True])
    )

In [3]:
# All 15 tables: batch size (outer) x condition (inner).
tables = {}
for bs in BATCH_DIRS:
    for cond_slug, (disp, init) in CONDITIONS.items():
        ep_df = epochs_to_target(bs, cond_slug, init, TARGET_ACC)
        tables[(bs, cond_slug)] = ep_df
        display(Markdown(
            f"### batch {bs} — condition `{disp}` (init: {init}) "
            f"— epochs to test acc ≥ {TARGET_ACC:.0%}"
        ))
        display(ep_df)

### batch 16 — condition `p only` (init: new) — epochs to test acc ≥ 80%

,n_seeds_reached,mean_epoch,std_epoch,seed_0,seed_1,seed_2,seed_3
task,,,,,,,
CatPro,2,174.5,13.5,188.0,161.0,None,None
NoiseCleaner,0,NaN,NaN,NaN,NaN,None,None
DelayPro,0,NaN,NaN,NaN,NaN,None,None
DelayAnti,0,NaN,NaN,NaN,NaN,None,None
CatAnti,0,NaN,NaN,NaN,NaN,None,None
Match2Sample,0,NaN,NaN,NaN,NaN,None,None
NonMatch2Sample,0,NaN,NaN,NaN,NaN,None,None
CtxIntMod1,0,NaN,NaN,NaN,NaN,None,None
CtxIntMod2,0,NaN,NaN,NaN,NaN,None,None


### batch 16 — condition `p + p2C` (init: random_peer) — epochs to test acc ≥ 80%

,n_seeds_reached,mean_epoch,std_epoch,seed_0,seed_1,seed_2,seed_3
task,,,,,,,
NoiseCleaner,4,0.000000,0.000000,0.0,0.0,0.0,0.0
PerceptualDM,4,0.000000,0.000000,0.0,0.0,0.0,0.0
CatPro,4,2.500000,3.201562,1.0,8.0,0.0,1.0
MultiSens,4,10.000000,3.741657,16.0,6.0,10.0,8.0
CatAnti,4,10.250000,5.068284,17.0,9.0,3.0,12.0
DelayPro,4,11.000000,1.870829,11.0,9.0,14.0,10.0
DelayedComparison,4,11.000000,4.743416,14.0,15.0,3.0,12.0
CtxIntMod2,4,11.250000,4.815340,15.0,14.0,3.0,13.0
CtxIntMod1,4,22.750000,11.605494,20.0,42.0,18.0,11.0


### batch 16 — condition `p + p2D` (init: random_peer) — epochs to test acc ≥ 80%

...

,n_seeds_reached,mean_epoch,std_epoch,seed_0,seed_1,seed_2,seed_3
task,,,,,,,
PerceptualDM,4,0.000000,0.000000,0.0,0.0,0.0,0.0
NoiseCleaner,4,0.750000,1.299038,0.0,0.0,3.0,0.0
CatPro,4,8.000000,2.121320,8.0,8.0,5.0,11.0
CtxIntMod1,4,16.250000,13.460591,39.0,4.0,12.0,10.0
IntDisc,4,23.250000,29.422568,8.0,2.0,74.0,9.0
CatAnti,4,23.500000,14.534442,22.0,44.0,3.0,25.0
DelayedComparison,4,39.500000,9.962429,33.0,54.0,28.0,43.0
ArithAdd,4,72.500000,19.576772,55.0,82.0,100.0,53.0
CtxIntMod2,4,91.250000,41.847192,77.0,153.0,98.0,37.0


### batch 64 — condition `p + p2C + p2D` (init: random_peer) — epochs to test acc ≥ 80%

,n_seeds_reached,mean_epoch,std_epoch,seed_0,seed_1,seed_2,seed_3
task,,,,,,,
NoiseCleaner,4,0.000000,0.000000,0.0,0.0,0.0,0.0
PerceptualDM,4,0.750000,1.299038,0.0,0.0,3.0,0.0
CatPro,4,2.500000,1.802776,2.0,5.0,0.0,3.0
CtxIntMod1,4,6.000000,4.847680,14.0,1.0,4.0,5.0
DelayedComparison,4,7.000000,5.787918,1.0,16.0,3.0,8.0
IntDisc,4,7.250000,4.322904,2.0,14.0,6.0,7.0
DelayPro,4,9.000000,1.000000,10.0,8.0,8.0,10.0
CatAnti,4,15.000000,7.035624,18.0,21.0,3.0,18.0
MultiSens,4,17.750000,8.437269,23.0,14.0,28.0,6.0


### batch 64 — condition `p + p2A + p2W` (init: random_peer) — epochs to test acc ≥ 80%

,n_seeds_reached,mean_epoch,std_epoch,seed_0,seed_1,seed_2,seed_3
task,,,,,,,
NoiseCleaner,4,0.00,0.000000,0.0,0.0,0.0,0.0
CatPro,4,0.50,0.866025,0.0,2.0,0.0,0.0
PerceptualDM,4,0.50,0.866025,0.0,0.0,2.0,0.0
CtxIntMod1,4,2.00,1.000000,3.0,1.0,3.0,1.0
CatAnti,4,3.75,1.299038,3.0,5.0,2.0,5.0
IntDisc,4,4.00,1.224745,3.0,4.0,6.0,3.0
DelayPro,4,4.75,1.479020,7.0,5.0,3.0,4.0
CtxIntMod2,4,6.25,2.487469,10.0,7.0,4.0,4.0
DelayAnti,4,8.25,1.299038,7.0,9.0,10.0,7.0


## Summary across conditions

How many of the 19 held-out tasks reach the target in **all 4 seeds**, and the
median first-crossing epoch over tasks that reach it (lower = faster transfer).

In [4]:
summary = []
for (bs, cond_slug), ep_df in tables.items():
    disp = CONDITIONS[cond_slug][0]
    all4 = int((ep_df["n_seeds_reached"] == len(SEEDS)).sum())
    any1 = int((ep_df["n_seeds_reached"] >= 1).sum())
    med_epoch = ep_df.loc[ep_df["n_seeds_reached"] >= 1, "mean_epoch"].median()
    summary.append(
        {
            "batch": bs,
            "condition": disp,
            "tasks_all4_seeds": all4,
            "tasks_any_seed": any1,
            "median_epoch_to_target": round(med_epoch, 1) if med_epoch == med_epoch else np.nan,
        }
    )

summary_df = pd.DataFrame(summary)
display(Markdown(f"**Target = {TARGET_ACC:.0%}; 19 held-out tasks, {len(SEEDS)} seeds**"))
summary_df

**Target = 80%; 19 held-out tasks, 4 seeds**

,batch,condition,tasks_all4_seeds,tasks_any_seed,median_epoch_to_target
0,16,p only,0,1,174.5
1,16,p + p2C,12,16,17.0
2,16,p + p2D,12,15,17.2
3,16,p + p2C + p2D,12,17,6.5
4,16,p + p2A + p2W,16,19,4.8
5,32,p only,0,0,NaN
6,32,p + p2C,12,15,20.2
7,32,p + p2D,11,15,23.8
8,32,p + p2C + p2D,13,17,8.0
9,32,p + p2A + p2W,15,19,7.5
